# 02. Basic Parsing and Guardrails

**Purpose:** Core **Prompt Engineering** notebook: baseline prompting, prompt tuning, and guardrails (**Task 1** and guardrails portion of **Task 2**).

**Three‑act structure**
- **Act I – The Baseline:** Run a starter prompt and obtain a baseline accuracy score.
- **Act II – The Power of Prompts:** Run a deliberately bad prompt, then edit/improve your prompt to beat the baseline.
- **Act III – Making It Robust:** Add “garbage” inputs and enable `use_guardrails` to filter invalid data.

**Key takeaway:** Instruction quality matters more than model tier; guardrails make systems reliable.


## Setup

Install dependencies and load shared utilities (same code as the original workshop notebooks).

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r esmt-workshop/requirements.txt


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report

print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ['WORKSHOP_EMAIL']

In [ ]:
# API access is controlled by email at the proxy server.
# No API key is required in this workshop implementation.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')

MODEL_NAME = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40
MAX_TOKENS = 512
MAX_WORKERS = 8


## Data

Load the labeled development set. We'll use a small slice for fast iteration, then run on the full set (with garbage inputs) for robustness checks.

In [ ]:
import pandas as pd

dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
dev_sample = dev_df.head(20).copy()

# Small set of "garbage" / non-address inputs to test guardrails.
garbage_data = pd.DataFrame([
    {'record_id': 'g1', 'Unstructured Address': 'book stores near me', 'Town Name': '', 'Postal Code': '', 'Country Code (2 characters)': ''},
    {'record_id': 'g2', 'Unstructured Address': 'what is the weather today?', 'Town Name': '', 'Postal Code': '', 'Country Code (2 characters)': ''},
])

dev_df_with_garbage = pd.concat([dev_df, garbage_data], ignore_index=True)

print('dev_df:', dev_df.shape, '| dev_sample:', dev_sample.shape, '| dev_df_with_garbage:', dev_df_with_garbage.shape)


## Prompt Cell

Edit prompts here. Keep JSON strict.

## Step 4: Define Reusable Experiment Function


In [ ]:
from esmt_workshop.student_api import process_batch_addresses


def run_experiment(prompt_template: str, *, temperature: float = 0.0, top_p: float = 1.0, top_k: int = 40):
    # Load development labels for prompt iteration.
    dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

    # Use the same model while changing prompt and sampling controls.
    pred_df = process_batch_addresses(
        dev_df,
        email=STUDENT_EMAIL,
        stage='prompt_tuned',
        model=MODEL_NAME,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        max_tokens=MAX_TOKENS,
        custom_prompt_template=prompt_template,
        max_workers=MAX_WORKERS,
    )

    report = evaluate_predictions(pred_df, dev_df, email=STUDENT_EMAIL)
    return pred_df, report


In [ ]:
BAD_PROMPT_TEMPLATE = "Parse this address: {address}. Give me JSON."

In [ ]:
# Starter prompt editable directly in code (same as original 02_prompt_tuning.ipynb).
STARTER_PROMPT_TEMPLATE = '''You are an address parser for AML compliance.

Input address:
{address}

Return strict JSON only using this schema:
{schema}

Rules:
1. Town Name is only city/town/locality.
2. Postal Code includes only postal token(s).
3. Remaining Address includes everything else.
4. Country Code is ISO alpha-2 uppercase.
5. Use empty strings when uncertain.
'''


In [ ]:
# Your prompt (edit this to beat the baseline) — same starting point as original.
STUDENT_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''


## Execute Pipeline on Limited Dataset (Experiments)

Run baseline, bad prompt, and your prompt on a small sample for fast feedback.

In [ ]:
# Reusable helper for running on an explicit dataframe (limited sample).
from esmt_workshop.student_api import process_batch_addresses
from esmt_workshop.evaluation import evaluate_predictions

def run_experiment_on_df(input_df: pd.DataFrame, prompt_template: str, *, use_guardrails: bool = False):
    pred_df = process_batch_addresses(
        input_df,
        email=STUDENT_EMAIL,
        stage='prompt_tuned',
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        max_tokens=MAX_TOKENS,
        use_guardrails=use_guardrails,
        custom_prompt_template=prompt_template,
        max_workers=MAX_WORKERS,
    )
    report = evaluate_predictions(pred_df, input_df, email=STUDENT_EMAIL)
    return pred_df, report


In [ ]:
# Act I — Baseline (starter prompt)
pred_starter_small, report_starter_small = run_experiment_on_df(dev_sample, STARTER_PROMPT_TEMPLATE, use_guardrails=False)

# Act II — Bad prompt
pred_bad_small, report_bad_small = run_experiment_on_df(dev_sample, BAD_PROMPT_TEMPLATE, use_guardrails=False)

# Act II — Your prompt
pred_student_small, report_student_small = run_experiment_on_df(dev_sample, STUDENT_PROMPT_TEMPLATE, use_guardrails=False)

report_student = {
    'pred_starter_small': pred_starter_small,
    'report_starter_small': report_starter_small,
    'pred_bad_small': pred_bad_small,
    'report_bad_small': report_bad_small,
    'pred_student_small': pred_student_small,
    'report_student_small': report_student_small,
}
print('Starter micro_accuracy:', report_starter_small['summary'].get('micro_accuracy'))
print('Bad micro_accuracy:', report_bad_small['summary'].get('micro_accuracy'))
print('Student micro_accuracy:', report_student_small['summary'].get('micro_accuracy'))


## Analysis (Limited Dataset)

Use `report_student[...]` to explore reports and dataframes.

In [ ]:
# Examples:
report_student['report_student_small']['summary']
report_student['report_student_small']['field_metrics']
report_student['report_student_small']['mismatches'].head(10)


## Execute Pipeline on Complete Dataset

Act III: add garbage inputs and compare runs with and without guardrails.

In [ ]:
pred_full_no_guard, report_full_no_guard = run_experiment_on_df(
    dev_df_with_garbage,
    STUDENT_PROMPT_TEMPLATE,
    use_guardrails=False,
)

pred_full_with_guard, report_full_with_guard = run_experiment_on_df(
    dev_df_with_garbage,
    STUDENT_PROMPT_TEMPLATE,
    use_guardrails=True,
)

report_student.update({
    'pred_full_no_guard': pred_full_no_guard,
    'report_full_no_guard': report_full_no_guard,
    'pred_full_with_guard': pred_full_with_guard,
    'report_full_with_guard': report_full_with_guard,
})
print('Full (no guardrails) micro_accuracy:', report_full_no_guard['summary'].get('micro_accuracy'))
print('Full (with guardrails) micro_accuracy:', report_full_with_guard['summary'].get('micro_accuracy'))


## Analysis (Complete Dataset)

Use `report_student[...]` to inspect full-run performance and guardrail behavior.

In [ ]:
# Examples:
report_student['report_full_with_guard']['summary']
report_student['report_full_with_guard']['mismatches'].head(10)

# Which inputs were rejected?
pred_full_with_guard = report_student['pred_full_with_guard']
if 'valid_input' in pred_full_with_guard.columns:
    rejected = pred_full_with_guard[pred_full_with_guard['valid_input'] == False]
    rejected.head(20)
else:
    print("No 'valid_input' column found in predictions (toolkit version may differ).")


## Publish to Leaderboard (Manual)

Run this **only when you are ready**. It writes `final_submission.csv` from the latest full run.

In [ ]:
PUBLISH_TO_LEADERBOARD = False  # set True to export final_submission.csv

if PUBLISH_TO_LEADERBOARD:
    out_dir = PROJECT_ROOT / 'outputs'
    out_dir.mkdir(parents=True, exist_ok=True)

    # Use the guardrails-enabled full run by default.
    pred_df = report_student.get('pred_full_with_guard')
    if pred_df is None:
        raise ValueError("Run the complete dataset section first.")

    # Try to produce a submission with record_id + predicted fields (toolkit may include extra cols).
    # Keep all columns to avoid accidentally dropping required outputs.
    submission_path = out_dir / 'final_submission.csv'
    pred_df.to_csv(submission_path, index=False)
    print("Wrote:", submission_path)
else:
    print("Set PUBLISH_TO_LEADERBOARD = True to export final_submission.csv")
